> **Step 1 session `acc4_slot2_subtype_B_ho400`: already configured, do not edit.** Kaggle account 4, slot 2, start **after the pilot finishes**. Settings: GPU T4, Internet ON. Add Input: breakhis. Expected about 7 h. Then Save & Run All.

# Subtype extension of the magnification-shift study: detection, then subtype classification (BreakHis)

The study's theme is unchanged: how accurate **and how reliable** CNN, Transformer, Mamba and foundation-model
encoders are when tested on a **magnification they never saw in training**. This notebook extends the task
from benign vs malignant to the eight BreakHis subtypes, in two stages:

* **Stage 1, detection:** benign vs malignant.
* **Stage 2, subtype classification:** two 4-class models, one for the benign subtypes (adenosis,
  fibroadenoma, phyllodes tumour, tubular adenoma) and one for the malignant subtypes (ductal, lobular,
  mucinous and papillary carcinoma). At test time stage 1's prediction decides which stage-2 model runs, and
  the pipeline is scored end to end on the 8 subtypes.

**Encoders** are the same four as the binary study: ConvNeXt-T and Swin-T (ImageNet-initialised, fine-tuned
end to end), MedMamba (random initialisation) and a linear probe on the frozen Phikon foundation model.
**Metrics** are the same too: accuracy, balanced accuracy and macro-F1 for accuracy, and ECE, Brier,
risk at 80% coverage and AURC for reliability, per stage and for the whole pipeline.

**Protocol (`PROTOCOL = 'lomo'`).** Leave-one-magnification-out, as in the binary study. The held-out
magnification is the test set; the other three are used for training and validation. `PROTOCOL = 'pooled'`
(all magnifications mixed) is kept only as a reference without magnification shift.

**Split (`SPLIT_MODE = 'image'`).** The three training magnifications are split into train and validation
image by image, stratified by subtype and magnification, and every image at the held-out magnification is a
test image. Test images are never seen in training, but **the same patients are seen at the other
magnifications**, so this measures robustness to magnification, not generalisation to new patients; the
notebook prints the overlap. `SPLIT_MODE = 'patient'` keeps patients apart (65 / 15 / 20 %, as in the binary
study), grouping slide ids by their numeric core (22549AB, 22549CD and 22549G are one patient). Rare subtypes
have only a few patients, so a class can then be missing from a seed's validation or test set.

**Class imbalance.** Each stage's cross-entropy loss is weighted by inverse class frequency in its training
split (ductal carcinoma has 3,451 images, adenosis 444).

## Run guide: 4 accounts x 2 sessions, all in parallel

One Kaggle account can run two GPU sessions at once. Give each account one held-out magnification and run
two sessions on it:

| Session | `MODELS_THIS_ACCOUNT` | `INCLUDE_FM` | `HELD_OUT_OVERRIDE` | Expected time (T4) |
|---|---|---|---|---|
| A | `['convnext_tiny', 'swin_tiny']` | `True` | the account's magnification, e.g. `[40]` | about 9.5 h |
| B | `['medmamba']` | `False` | the same magnification | about 7 h |

Accounts 1-4 use `[40]`, `[100]`, `[200]` and `[400]`. That is 8 sessions and covers all four held-out
magnifications x 3 seeds x 3 stages for every encoder.

**Ablation, one more session** (on any account once one of its sessions has finished):

| Session | `MODELS_THIS_ACCOUNT` | `INCLUDE_FM` | `HELD_OUT_OVERRIDE` | Expected time (T4) |
|---|---|---|---|---|
| C | `['medmamba_no_ssm']` | `False` | `[400]` | about 4 h |

As in the binary study, every SS2D (2-D selective scan) block of MedMamba is replaced by a depthwise-conv block
of the same width, and the result is compared with MedMamba at held-out 400x (same seeds and split), stage by
stage and end to end.

1. **File -> Import Notebook**, then set the variables in the protocol cell for your session.
2. Settings: **GPU T4**, **Internet ON**. Add Input: `ambarish/breakhis`.
3. **Save & Run All.** If a commit stops at the time budget, add that version as an input and run again:
   finished runs are restored and skipped.

Each stage run writes `runs/<run_id>/record.json`, `best.pt` (or `probe.pt`) and `test_outputs.npz`. When all
three stages of an encoder, magnification and seed are finished, the last cell scores the pipeline and writes
`hierarchical_results.csv`, per-class and per-magnification tables and 8 x 8 confusion matrices.

**Afterwards:** attach every session's final output to `analysis_notebook_v3.ipynb`. Its section 4b finds them
automatically, re-scores all sessions together with the same scoring code as this notebook, and draws
Fig 32-37 and tables T14-T17.

In [ ]:
# ---------- Protocol. Set the first four variables for your session (see the run guide), then Run All ----------
MODELS_THIS_ACCOUNT = ['medmamba']   # set for session acc4_slot2_subtype_B_ho400
                                                       # 'medmamba_no_ssm' (the SS2D ablation)
INCLUDE_FM = False   # set for session acc4_slot2_subtype_B_ho400
HELD_OUT_OVERRIDE = [400]   # set for session acc4_slot2_subtype_B_ho400
SPLIT_MODE = 'image'                           # 'image' (main result) or 'patient' (no patient in two splits)
PROTOCOL = 'lomo'                              # 'lomo' = the study's magnification-shift protocol; 'pooled' = all
                                               #  magnifications mixed (no magnification shift; kept for reference)

SEEDS = [13, 42, 2025]
RUN_NAME = None                                # None = derived from the settings above
BREAKHIS_ROOT = None                           # None = auto-discover under /kaggle/input
SOURCE_ROOT = '/kaggle/working/mamba_sources'

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
PROBE_EPOCHS = 200                             # Phikon linear probe, trained on cached features
PROBE_LR = 1e-3
PROBE_BATCH = 256
# LOMO, image-level: the held-out magnification is the test set; the other three are split into train / val.
LOMO_IMAGE_VAL_RATIO = 0.15
# LOMO, patient-level: patients (numeric cores) go 65 / 15 / 20 to train / val / test, as in the binary study.
LOMO_PATIENT_RATIOS = {'val': 0.15, 'test': 0.20}
# Pooled protocol only.
POOLED_RATIOS = {'train': 0.70, 'val': 0.15, 'test': 0.15}
CLASS_WEIGHTING = 'inverse_frequency'          # or None for an unweighted loss
TEMPERATURE_SCALE = True
MAX_RUNTIME_HOURS = 11.5
FM_NAME = 'owkin/phikon'
MAGNIFICATIONS = [40, 100, 200, 400]
CHECKPOINT_RULE = 'maximum validation macro-F1; earliest epoch breaks ties; test evaluated once after selection'

assert PROTOCOL in ('lomo', 'pooled'), PROTOCOL
assert SPLIT_MODE in ('image', 'patient'), SPLIT_MODE
assert set(MODELS_THIS_ACCOUNT) <= {'convnext_tiny', 'swin_tiny', 'medmamba', 'medmamba_no_ssm'}, MODELS_THIS_ACCOUNT
assert MODELS_THIS_ACCOUNT or INCLUDE_FM, 'nothing to train: add a model or set INCLUDE_FM = True'
HELD_OUT_MAGS = [int(m) for m in HELD_OUT_OVERRIDE] if PROTOCOL == 'lomo' else [None]
assert PROTOCOL == 'pooled' or set(HELD_OUT_MAGS) <= set(MAGNIFICATIONS), HELD_OUT_MAGS
if RUN_NAME is None:
    RUN_NAME = ('subtype_' + (f"ho{'-'.join(str(m) for m in HELD_OUT_MAGS)}" if PROTOCOL == 'lomo' else 'pooled')
                + f'_{SPLIT_MODE}_' + ('-'.join(MODELS_THIS_ACCOUNT) or 'none') + ('_fm' if INCLUDE_FM else ''))
OUTPUT = f'/kaggle/working/{RUN_NAME}'
FROM_SCRATCH_MODELS = list(MODELS_THIS_ACCOUNT)

print(f'RUN_NAME    = {RUN_NAME}')
print(f'OUTPUT      = {OUTPUT}')
print(f'PROTOCOL    = {PROTOCOL}' + (f'   held-out magnification(s): {HELD_OUT_MAGS}' if PROTOCOL == 'lomo' else ''))
print(f'MODELS      = {FROM_SCRATCH_MODELS} (+ Phikon probe: {INCLUDE_FM})')
print(f'SPLIT_MODE  = {SPLIT_MODE}   SEEDS = {SEEDS}   EPOCHS = {EPOCHS}   class weighting = {CLASS_WEIGHTING}')

In [ ]:
# ---------- Clear stale local runs, then set up the environment ----------
# On a "Save & Run All" commit /kaggle/working starts empty, so the clean-up is a no-op there; it matters only
# in an interactive session with persistence. Finished runs from an attached earlier version are restored by
# the resume cell, which checks that they were trained on this exact manifest and split mode.
import os, sys, json, random, time, hashlib, platform, subprocess, shutil, re
from pathlib import Path
from datetime import datetime, timezone

RESET_STATE = True
if RESET_STATE:
    for pth in [Path(OUTPUT, 'runs'), Path(OUTPUT, 'raw_runs.csv'), Path(OUTPUT, 'hierarchical_results.csv')]:
        if pth.exists():
            shutil.rmtree(pth) if pth.is_dir() else pth.unlink()
            print('deleted', pth)

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
Path(OUTPUT).mkdir(parents=True, exist_ok=True)
Path(SOURCE_ROOT).mkdir(parents=True, exist_ok=True)


def pip_install_if_missing(module_name, spec=None):
    try:
        __import__(module_name); return True
    except ImportError:
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec or module_name], check=True)
            return True
        except subprocess.CalledProcessError as exc:
            print(f'Install failed for {spec or module_name}:', exc); return False


for module_name, spec in [('timm', 'timm'), ('einops', 'einops'), ('transformers', 'transformers'),
                          ('fvcore', 'fvcore')]:
    pip_install_if_missing(module_name, spec)

# The compiled selective-scan kernel is needed only for MedMamba (and its no-SS2D ablation, built from it).
MAMBA_SSM_OK = False
if any(m.startswith('medmamba') for m in FROM_SCRATCH_MODELS):
    try:
        import mamba_ssm; MAMBA_SSM_OK = True
        print('mamba_ssm already installed:', mamba_ssm.__version__)
    except ImportError:
        for spec in [['mamba-ssm', 'causal-conv1d'], ['mamba-ssm==2.2.4', 'causal-conv1d==1.5.0.post8'],
                     ['mamba-ssm==2.2.2', 'causal-conv1d==1.4.0']]:
            try:
                print('Trying compiled mamba_ssm install:', spec)
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation'] + spec, check=True)
                import importlib; mamba_ssm = importlib.import_module('mamba_ssm'); MAMBA_SSM_OK = True
                print(' -> loaded mamba_ssm:', mamba_ssm.__version__); break
            except Exception as exc:
                print(' -> failed:', str(exc)[:180])

import numpy as np, pandas as pd, torch, torch.nn as nn
import timm, torchvision, sklearn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, recall_score, precision_score)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

assert torch.cuda.is_available(), 'Kaggle GPU accelerator is required (Settings -> Accelerator -> GPU T4).'
DEVICE = torch.device('cuda:0')


def _reset_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def _peak_mib():
    return torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else float('nan')


def command_text(args):
    try: return subprocess.check_output(args, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as exc: return str(exc)


env = {'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'python': sys.version,
       'platform': platform.platform(), 'torch': torch.__version__, 'torch_cuda': torch.version.cuda,
       'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
       'timm': timm.__version__, 'torchvision': torchvision.__version__, 'sklearn': sklearn.__version__,
       'mamba_ssm_ok': MAMBA_SSM_OK, 'pip_freeze': command_text([sys.executable, '-m', 'pip', 'freeze'])}
Path(OUTPUT, 'environment.json').write_text(json.dumps(env, indent=2, default=str))
config = {k: v for k, v in globals().items()
          if k.isupper() and k not in {'OUTPUT', 'SOURCE_ROOT', 'DEVICE', 'BREAKHIS_ROOT'}}
Path(OUTPUT, 'config.json').write_text(json.dumps(config, indent=2, default=str))
print(json.dumps({k: env[k] for k in ('gpu_name', 'torch', 'torch_cuda', 'timm', 'mamba_ssm_ok')}, indent=2))

In [ ]:
# ---------- MedMamba source, pinned to the same commit as the magnification-shift study ----------
EXTERNAL_FACTORIES = {}
source_manifest = {}


def clone_at(name, url, commit=None):
    destination = Path(SOURCE_ROOT, name)
    if not destination.exists():
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    if commit is not None:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    return destination


_MAMBA_FAMILY = [m for m in FROM_SCRATCH_MODELS if m.startswith('medmamba')]
if _MAMBA_FAMILY:
    if not MAMBA_SSM_OK:
        print('mamba_ssm unavailable: MedMamba cannot run; removed from this session:', _MAMBA_FAMILY)
        for _m in _MAMBA_FAMILY:
            FROM_SCRATCH_MODELS.remove(_m)
    else:
        try:
            med_src = clone_at('MedMamba', 'https://github.com/YubiaoYue/MedMamba.git',
                               commit='104beb3d641d67cef6f09b94ac299083d91bc417')
            safe_med = Path(SOURCE_ROOT, 'medmamba_safe.py')
            source = Path(med_src, 'MedMamba.py').read_text()
            marker = '\nmedmamba_t = VSSM'                 # drop the module-level demo model
            safe_med.write_text(source.split(marker, 1)[0] + '\n' if marker in source else source)
            import importlib.util
            spec = importlib.util.spec_from_file_location('medmamba_safe', safe_med)
            med_module = importlib.util.module_from_spec(spec); spec.loader.exec_module(med_module)
            MedMambaVSSM = med_module.VSSM
            EXTERNAL_FACTORIES['medmamba'] = lambda n: MedMambaVSSM(depths=[2, 2, 4, 2], dims=[96, 192, 384, 768],
                                                                    num_classes=n)
            source_manifest['MedMamba_commit'] = command_text(['git', '-C', str(med_src), 'rev-parse', 'HEAD'])
            Path(OUTPUT, 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2))
            print('MedMamba registered at', source_manifest['MedMamba_commit'][:12])
        except Exception as exc:
            print('MedMamba registration failed, removed from this session:', repr(exc)[:200])
            for _m in _MAMBA_FAMILY:
                FROM_SCRATCH_MODELS.remove(_m)

# ---- ablation: MedMamba with every SS2D (2-D selective scan) replaced by a depthwise-conv block ----
# Copied from the binary study's Part C notebook, where this variant built and trained on Kaggle
# (10.91 M parameters, 1.65 GFLOPs). Same comparison as there: its delta to MedMamba at the same held-out
# magnification, seeds and split measures what the state-space component contributes.
import torch.nn as nn


class NoSSMBlock(nn.Module):
    """Drop-in replacement for SS2D (NHWC in, NHWC out): LayerNorm -> 3x3 depthwise conv -> 1x1 conv -> SiLU."""
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dwconv = nn.Conv2d(d_model, d_model, 3, padding=1, groups=d_model, bias=True)
        self.pwconv = nn.Conv2d(d_model, d_model, 1, bias=True)
        self.act = nn.SiLU()

    def forward(self, x, *args, **kwargs):
        if x.dim() == 4:                                   # (B, H, W, C), as MedMamba's SS2D receives it
            y = self.norm(x).permute(0, 3, 1, 2)           # (B, C, H, W)
            return self.act(self.pwconv(self.dwconv(y))).permute(0, 2, 3, 1)
        if x.dim() == 3:                                   # (B, N, C)
            B, N, C = x.shape
            hw = int(N ** 0.5)
            y = self.norm(x).transpose(1, 2).reshape(B, C, hw, hw)
            return self.act(self.pwconv(self.dwconv(y))).reshape(B, C, N).transpose(1, 2)
        return self.act(self.pwconv(self.dwconv(x)))


def _ss2d_channels(mod):
    if isinstance(getattr(mod, 'd_model', None), int) and mod.d_model > 0:
        return mod.d_model
    if hasattr(mod, 'in_proj') and hasattr(mod.in_proj, 'in_features'):
        return int(mod.in_proj.in_features)
    if hasattr(mod, 'out_proj') and hasattr(mod.out_proj, 'out_features'):
        return int(mod.out_proj.out_features)
    raise RuntimeError('cannot infer the channel width of an SS2D module')


def build_medmamba_no_ssm(num_classes):
    model = EXTERNAL_FACTORIES['medmamba'](num_classes)
    n = 0
    for parent in model.modules():
        for child_name, child in list(parent.named_children()):
            if 'SS2D' in child.__class__.__name__:
                setattr(parent, child_name, NoSSMBlock(_ss2d_channels(child))); n += 1
    if n == 0:
        raise RuntimeError('no SS2D module found in MedMamba; the ablation would be a no-op')
    return model


if 'medmamba_no_ssm' in FROM_SCRATCH_MODELS and 'medmamba' in EXTERNAL_FACTORIES:
    EXTERNAL_FACTORIES['medmamba_no_ssm'] = build_medmamba_no_ssm
    print('medmamba_no_ssm registered (SS2D -> depthwise conv).')
print('Active trained encoders:', FROM_SCRATCH_MODELS, '| Phikon probe:', INCLUDE_FM)

In [ ]:
# ---------- BreakHis manifest, the two stages, and the train / validation / test split ----------
# The file-name pattern is the corrected one from the magnification-shift study: slide ids are alphanumeric
# (22549AB, 16184CD), and the earlier purely numeric pattern silently dropped 2,117 images.
NAME = re.compile(r'^(?P<patient>SOB_(?P<coarse>[BM])_(?P<subtype>[A-Z]+)-\d+-[A-Za-z0-9]+)-(?P<magnification>\d+)-\d+', re.I)
SUBTYPE_TO_ID = {'A': 0, 'F': 1, 'PT': 2, 'TA': 3, 'DC': 4, 'LC': 5, 'MC': 6, 'PC': 7}
SUBTYPE_NAMES = {'A': 'adenosis', 'F': 'fibroadenoma', 'PT': 'phyllodes tumour', 'TA': 'tubular adenoma',
                 'DC': 'ductal carcinoma', 'LC': 'lobular carcinoma', 'MC': 'mucinous carcinoma',
                 'PC': 'papillary carcinoma'}
BENIGN_SUBTYPES = ['A', 'F', 'PT', 'TA']          # stage-2 benign model, local labels 0..3 = global 0..3
MALIGNANT_SUBTYPES = ['DC', 'LC', 'MC', 'PC']     # stage-2 malignant model, local labels 0..3 = global 4..7


def discover_breakhis_root(search_root='/kaggle/input'):
    best, best_count = None, 0
    root = Path(search_root)
    candidates = ([p for p in sorted(root.iterdir()) if p.is_dir()] + [root]) if root.exists() else []
    for candidate in candidates:
        try:
            count = sum(1 for _ in candidate.rglob('SOB_*.png'))
        except OSError:
            continue
        if count > best_count:
            best, best_count = candidate, count
    if best is None:
        raise RuntimeError('BreakHis not found. Add Input -> ambarish/breakhis, or set BREAKHIS_ROOT.')
    print('BREAKHIS_ROOT =', best, f'({best_count} SOB files)')
    return str(best)


def build_manifest(root):
    rows, seen = [], set()
    for path in sorted(Path(root).rglob('SOB_*.png')):
        match = NAME.match(path.name)
        if not match or path.name in seen:              # a mirror can hold the same file twice
            continue
        seen.add(path.name)
        item = match.groupdict()
        subtype = item['subtype'].upper()
        if subtype not in SUBTYPE_TO_ID:
            continue
        rows.append({'path': str(path), 'patient_id': item['patient'], 'magnification': int(item['magnification']),
                     'coarse_label': int(item['coarse'].upper() == 'M'), 'subtype': subtype,
                     'subtype_label': SUBTYPE_TO_ID[subtype]})
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError('No BreakHis images found under ' + str(root))
    # 82 slide ids reduce to 70 numeric cores (22549AB / 22549CD / 22549G ...). The core is the patient unit.
    frame['core_id'] = frame.patient_id.str.extract(r'-(\d+-\d+)[A-Za-z]*$')[0]
    assert frame.core_id.notna().all(), 'a slide id did not yield a numeric core'
    return frame.reset_index(drop=True)


if BREAKHIS_ROOT is None or not list(Path(BREAKHIS_ROOT).rglob('SOB_*.png')):
    BREAKHIS_ROOT = discover_breakhis_root()
MANIFEST = build_manifest(BREAKHIS_ROOT)
MANIFEST.to_csv(Path(OUTPUT, 'breakhis_manifest.csv'), index=False)
print(f'images {len(MANIFEST)} | slide ids {MANIFEST.patient_id.nunique()} | patient cores {MANIFEST.core_id.nunique()}')
print(MANIFEST.pivot_table(index='subtype', columns='magnification', values='path', aggfunc='count')
      .reindex(BENIGN_SUBTYPES + MALIGNANT_SUBTYPES).to_string())

# ---- the three models of the pipeline ----
STAGES = {
    'detect': {'classes': ['benign', 'malignant'],
               'select': lambda f: f,
               'label': lambda f: f.coarse_label.astype(int)},
    'benign_subtype': {'classes': BENIGN_SUBTYPES,
                       'select': lambda f: f[f.coarse_label == 0],
                       'label': lambda f: f.subtype.map({s: i for i, s in enumerate(BENIGN_SUBTYPES)}).astype(int)},
    'malignant_subtype': {'classes': MALIGNANT_SUBTYPES,
                          'select': lambda f: f[f.coarse_label == 1],
                          'label': lambda f: f.subtype.map({s: i for i, s in enumerate(MALIGNANT_SUBTYPES)}).astype(int)},
}
STAGE_ORDER = ['detect', 'benign_subtype', 'malignant_subtype']


def _grouped_three_way(idx, y, g, test_ratio, val_ratio, seed):
    """Patient cores to test / val / train with stratification by subtype (StratifiedGroupKFold)."""
    outer = StratifiedGroupKFold(n_splits=int(round(1 / test_ratio)), shuffle=True, random_state=seed)
    rest_i, te_i = next(outer.split(idx, y, g))
    inner = StratifiedGroupKFold(n_splits=int(round((1 - test_ratio) / val_ratio)), shuffle=True, random_state=seed)
    tr_j, va_j = next(inner.split(idx[rest_i], y[rest_i], g[rest_i]))
    return idx[rest_i][tr_j], idx[rest_i][va_j], idx[te_i]


def make_split(frame, seed, held_out, mode=SPLIT_MODE):
    """One split per (held-out magnification, seed), shared by all three stages so the pipeline is scored on
    one test set. Rows outside the protocol get split = 'drop' and keep their manifest index."""
    frame = frame.copy()
    idx = np.arange(len(frame))
    strata = (frame.subtype + '_' + frame.magnification.astype(str)).to_numpy()
    y, g = frame.subtype_label.to_numpy(), frame.core_id.to_numpy()
    if held_out is None:                                            # pooled protocol
        r = POOLED_RATIOS
        if mode == 'image':
            hold = r['val'] + r['test']
            tr, rest = train_test_split(idx, test_size=hold, stratify=strata, random_state=seed)
            va, te = train_test_split(rest, test_size=r['test'] / hold, stratify=strata[rest], random_state=seed)
        else:
            tr, va, te = _grouped_three_way(idx, y, g, r['test'], r['val'], seed)
    else:                                                           # LOMO
        on_held_out = frame.magnification.to_numpy() == held_out
        if mode == 'image':
            # test = every image at the held-out magnification; the other three are split image by image
            pool = idx[~on_held_out]
            tr, va = train_test_split(pool, test_size=LOMO_IMAGE_VAL_RATIO, stratify=strata[pool], random_state=seed)
            te = idx[on_held_out]
        else:
            # patient-disjoint LOMO as in the binary study: patients to train / val / test, then train and val
            # use the three training magnifications and test uses the held-out one
            tr_all, va_all, te_all = _grouped_three_way(idx, y, g, LOMO_PATIENT_RATIOS['test'],
                                                        LOMO_PATIENT_RATIOS['val'], seed)
            tr, va, te = (np.intersect1d(tr_all, idx[~on_held_out]), np.intersect1d(va_all, idx[~on_held_out]),
                          np.intersect1d(te_all, idx[on_held_out]))
    frame['split'] = 'drop'
    frame.loc[frame.index[tr], 'split'] = 'train'
    frame.loc[frame.index[va], 'split'] = 'val'
    frame.loc[frame.index[te], 'split'] = 'test'
    return frame


def key_name(key):
    held_out, seed = key
    return (f'ho{held_out}' if held_out is not None else 'pooled') + f'_seed{seed}'


def describe_split(frame, key):
    counts = frame.pivot_table(index='subtype', columns='split', values='path', aggfunc='count', fill_value=0)
    counts = counts.reindex(index=BENIGN_SUBTYPES + MALIGNANT_SUBTYPES, columns=['train', 'val', 'test'], fill_value=0)
    cores = {s: set(frame.loc[frame.split == s, 'core_id']) for s in ('train', 'val', 'test')}
    shared = cores['test'] & (cores['train'] | cores['val'])
    note = ''
    if shared and SPLIT_MODE == 'image':
        note = ('   <- expected: image-level LOMO tests unseen images of patients seen at other magnifications'
                if key[0] is not None else '   <- expected with an image-level split')
    print(f'{key_name(key)}: images per split '
          f'{frame.split.value_counts().reindex(["train", "val", "test"], fill_value=0).to_dict()}')
    print(f'  test patients (cores) also seen in train/val: {len(shared)} of {len(cores["test"])}{note}')
    missing = [(s, sub) for sub in counts.index for s in ('val', 'test') if counts.loc[sub, s] == 0]
    if missing:
        print('  WARNING: subtype missing from a split (its per-class metrics will be undefined):', missing)
    return counts


SPLIT_KEYS = [(h, s) for h in HELD_OUT_MAGS for s in SEEDS]      # (held-out magnification or None, seed)
SPLITS, _split_counts = {}, {}
for key in SPLIT_KEYS:
    SPLITS[key] = make_split(MANIFEST, key[1], key[0])
    if key[0] is not None:
        assert not ((SPLITS[key].split.isin(['train', 'val'])) &
                    (SPLITS[key].magnification == key[0])).any(), 'held-out magnification leaked into training'
        assert ((SPLITS[key].split == 'test') <= (SPLITS[key].magnification == key[0])).all()
    if SPLIT_MODE == 'patient':
        used = SPLITS[key][SPLITS[key].split != 'drop']
        assert (used.groupby('core_id').split.nunique() == 1).all(), 'patient leakage: a core appears in two splits'
    _split_counts[key] = describe_split(SPLITS[key], key)
    SPLITS[key][['path', 'patient_id', 'core_id', 'magnification', 'subtype', 'split']].assign(
        path=lambda f: f.path.map(lambda p: Path(p).name)).to_csv(Path(OUTPUT, f'split_{key_name(key)}.csv'),
                                                                  index=False)
print(f'\nimages per subtype and split, {key_name(SPLIT_KEYS[0])}:')
print(_split_counts[SPLIT_KEYS[0]].to_string())


class HistologyDataset(Dataset):
    """rows must carry the stage label in column 'y'. Same transforms as the magnification-shift study."""
    def __init__(self, rows, train, mean=(.485, .456, .406), std=(.229, .224, .225)):
        self.rows = rows.reset_index(drop=True)
        ops = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
        if train:
            ops += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                    transforms.RandomRotation(15), transforms.ColorJitter(.1, .1, .05)]
        self.transform = transforms.Compose(ops + [transforms.ToTensor(), transforms.Normalize(mean, std)])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        return self.transform(Image.open(row.path).convert('RGB')), int(row.y), int(row.name_idx)

In [ ]:
# ---------- Resume, model factory and the frozen Phikon features ----------
# Resume: a commit that stopped at the time budget is continued by attaching that version as an input. Runs
# are restored only if they were trained on this exact manifest; the split is re-derived from the seed, and
# the stored split files are compared with it before anything is reused.
RESUME_FROM_INPUT = True


def _manifest_fingerprint(frame):
    view = pd.DataFrame({'file': frame['path'].map(lambda p: Path(str(p)).name),
                         'patient_id': frame['patient_id'].astype(str),
                         'magnification': frame['magnification'].astype(int),
                         'subtype': frame['subtype'].astype(str)})
    return hashlib.sha256(view.sort_values(['file', 'magnification']).to_csv(index=False).encode()).hexdigest()


if RESUME_FROM_INPUT and Path('/kaggle/input').exists():
    _here = _manifest_fingerprint(MANIFEST)
    _prev = sorted({c for c in Path('/kaggle/input').rglob(RUN_NAME) if c.is_dir() and (c / 'runs').is_dir()})
    _resumed = 0
    for _src in _prev:
        if not (_src / 'breakhis_manifest.csv').exists() or \
                _manifest_fingerprint(pd.read_csv(_src / 'breakhis_manifest.csv')) != _here:
            print(f'  REFUSED {_src}: different or missing manifest'); continue
        _same_split = all((_src / f'split_{key_name(k)}.csv').exists() and
                          pd.read_csv(_src / f'split_{key_name(k)}.csv').split.tolist() == SPLITS[k].split.tolist()
                          for k in SPLIT_KEYS)
        if not _same_split:
            print(f'  REFUSED {_src}: its split files differ from this session'); continue
        for _rec in sorted((_src / 'runs').glob('*/record.json')):
            _dst = Path(OUTPUT, 'runs', _rec.parent.name)
            if not (_dst / 'record.json').exists():
                shutil.copytree(_rec.parent, _dst, dirs_exist_ok=True); _resumed += 1
        print(f'  resumed runs from {_src}')
    print(f'Resume: {_resumed} finished runs restored.' if _prev else f'No earlier output named {RUN_NAME!r}; starting fresh.')


# ---- trained encoders: same builders as the magnification-shift study ----
def build_from_scratch(name, num_classes):
    if name == 'convnext_tiny':
        return timm.create_model('convnext_tiny', pretrained=True, num_classes=num_classes)
    if name == 'swin_tiny':
        return timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=num_classes)
    if name in EXTERNAL_FACTORIES:
        return EXTERNAL_FACTORIES[name](num_classes)
    raise KeyError(f'Unknown or unconfigured model: {name}')


compute_rows = []
for name in list(FROM_SCRATCH_MODELS):
    try:
        m = build_from_scratch(name, 2)
        compute_rows.append({'model': name, 'parameters': int(sum(p.numel() for p in m.parameters()))})
        del m; torch.cuda.empty_cache()
    except Exception as exc:
        print(f'Build failed for {name}, removed from this session:', repr(exc)[:200])
        FROM_SCRATCH_MODELS.remove(name)
if compute_rows:
    pd.DataFrame(compute_rows).to_csv(Path(OUTPUT, 'model_parameters.csv'), index=False)
    print(pd.DataFrame(compute_rows).to_string(index=False))

# ---- frozen Phikon: CLS features for every image, once ----
# The probe is trained on these cached features. They are extracted with the evaluation transform (resize +
# normalise, no augmentation), which is the usual linear-probe protocol; the magnification-shift study instead
# encoded one augmented view per training image. Stated here so the two protocols are never confused.
FM_FEATS = FM_DIM = None


class LinearProbe(nn.Module):
    def __init__(self, feature_dim, num_classes):
        super().__init__(); self.head = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        return self.head(x)


if INCLUDE_FM:
    try:
        from transformers import AutoModel
        _fm = AutoModel.from_pretrained(FM_NAME).eval().to(DEVICE)
        for _p in _fm.parameters():
            _p.requires_grad_(False)
        _all = MANIFEST.assign(y=0, name_idx=MANIFEST.index)
        _loader = DataLoader(HistologyDataset(_all, False), batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
        _feats, _order = [], []
        _t0 = time.perf_counter()
        with torch.inference_mode():
            for x, _, ni in _loader:
                out = _fm(pixel_values=x.to(DEVICE, non_blocking=True))
                _feats.append(out.last_hidden_state[:, 0].float().cpu()); _order.append(ni)
        FM_FEATS = torch.cat(_feats).numpy()                                    # (N, D), row i = MANIFEST row i
        assert np.array_equal(torch.cat(_order).numpy(), np.arange(len(MANIFEST)))
        FM_DIM = FM_FEATS.shape[1]
        del _fm; torch.cuda.empty_cache()
        print(f'Phikon CLS features {FM_FEATS.shape} in {time.perf_counter() - _t0:.0f} s')
    except Exception as exc:
        print('Phikon unavailable, probe branch skipped:', repr(exc)[:200])
        INCLUDE_FM = False

In [ ]:
# ---------- Metrics, class weights and the training of one stage model ----------
_trapz = getattr(np, 'trapezoid', None) or np.trapz


def softmax_np(logits, temperature=1.0):
    z = np.asarray(logits, dtype=np.float64) / float(temperature)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def expected_calibration_error(probs, labels, n_bins=15):
    conf, pred = probs.max(axis=1), probs.argmax(axis=1)
    acc = (pred == labels).astype(float)
    bins, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def risk_coverage(probs, labels):
    order = np.argsort(-probs.max(axis=1))
    err = (probs.argmax(axis=1) != labels).astype(float)[order]
    coverage = np.arange(1, len(err) + 1) / len(err)
    risk = np.cumsum(err) / np.arange(1, len(err) + 1)
    return coverage, risk, float(_trapz(risk, coverage))


_METRIC_KEYS = ['accuracy', 'balanced_accuracy', 'macro_f1', 'ece', 'brier', 'auc', 'risk_coverage_auc',
                'risk_at_coverage_80', 'risk_at_coverage_90', 'risk_at_coverage_95']


def compute_all_metrics(logits, labels, temperature=1.0):
    labels = np.asarray(labels).astype(int)
    if len(labels) == 0:                                  # e.g. a class-restricted stage with an empty split
        return {**{k: float('nan') for k in _METRIC_KEYS}, 'temperature': float(temperature)}, np.zeros((0, 0))
    probs = softmax_np(logits, temperature)
    k = probs.shape[1]
    pred = probs.argmax(axis=1)
    m = {'accuracy': accuracy_score(labels, pred), 'balanced_accuracy': balanced_accuracy_score(labels, pred),
         'macro_f1': f1_score(labels, pred, average='macro', labels=list(range(k)), zero_division=0),
         'ece': expected_calibration_error(probs, labels),
         'brier': float(((probs - np.eye(k)[labels]) ** 2).sum(axis=1).mean())}
    try:
        m['auc'] = (roc_auc_score(labels, probs[:, 1]) if k == 2 else
                    roc_auc_score(labels, probs, multi_class='ovr', average='macro', labels=list(range(k))))
    except ValueError:                                   # a class absent from this test set
        m['auc'] = float('nan')
    coverage, risk, rc_auc = risk_coverage(probs, labels)
    m['risk_coverage_auc'] = rc_auc
    for t in (0.80, 0.90, 0.95):
        i = int(t * len(coverage)) - 1
        m[f'risk_at_coverage_{int(100 * t)}'] = float(risk[i]) if i >= 0 else float('nan')
    m['temperature'] = float(temperature)
    return m, probs


def class_weights(labels, k):
    """Inverse class frequency in the training split: w_c = N / (k * n_c). A missing class gets weight 0."""
    counts = np.bincount(np.asarray(labels, dtype=int), minlength=k).astype(float)
    w = np.where(counts > 0, counts.sum() / (k * np.maximum(counts, 1)), 0.0)
    if (counts == 0).any():
        print(f'  WARNING: class(es) {np.where(counts == 0)[0].tolist()} absent from this training split')
    return w


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__(); self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        return logits / torch.exp(self.log_temp)


def fit_temperature(val_logits, val_labels, max_iter=200):
    scaler = TemperatureScaler().to(DEVICE)
    logits = torch.as_tensor(val_logits, device=DEVICE, dtype=torch.float32)
    labels = torch.as_tensor(val_labels, device=DEVICE, dtype=torch.long)
    optim = torch.optim.LBFGS([scaler.log_temp], lr=0.1, max_iter=max_iter)
    loss_fn = nn.CrossEntropyLoss()

    def closure():
        optim.zero_grad(); loss = loss_fn(scaler(logits), labels); loss.backward(); return loss
    optim.step(closure)
    return float(torch.exp(scaler.log_temp).item())


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark = True


def stage_rows(key, stage):
    """train / val / test rows of one stage for key = (held-out magnification or None, seed), with the stage
    label in 'y' and the manifest row in 'name_idx'."""
    frame = SPLITS[key].assign(name_idx=SPLITS[key].index)
    sel = STAGES[stage]['select'](frame)
    sel = sel.assign(y=STAGES[stage]['label'](sel))
    out = {s: sel[sel.split == s] for s in ('train', 'val', 'test')}
    for s in ('val', 'test'):
        if len(out[s]) == 0:
            print(f'  WARNING: {stage} has no {s} images for {key_name(key)}'
                  + (' - the last epoch is kept and T = 1' if s == 'val' else ' - its own test metrics are NaN'))
    assert len(out['train']), f'{stage} has no training images for {key_name(key)}'
    return out


def full_test_rows(key):
    """Every test image, whatever its class: the stage-2 models must score all of them for routing."""
    frame = SPLITS[key].assign(name_idx=SPLITS[key].index)
    return frame[frame.split == 'test'].assign(y=0)


def _loader(rows, train, batch=None):
    return DataLoader(HistologyDataset(rows, train), batch_size=batch or BATCH_SIZE, shuffle=train,
                      num_workers=NUM_WORKERS, pin_memory=True)


@torch.inference_mode()
def logits_labels_index(model, loader):
    model.eval()
    if len(loader.dataset) == 0:
        return np.zeros((0, 0)), np.zeros(0, dtype=int), np.zeros(0, dtype=int)
    lg, lb, ix = [], [], []
    for x, y, ni in loader:
        lg.append(model(x.to(DEVICE, non_blocking=True)).float().cpu().numpy())
        lb.append(np.asarray(y)); ix.append(np.asarray(ni))
    return np.concatenate(lg), np.concatenate(lb), np.concatenate(ix)


RECORD_COLUMNS = ['run_id', 'branch', 'model', 'stage', 'protocol', 'held_out_mag', 'split_mode', 'seed', 'num_classes',
                  'n_train', 'n_val', 'n_test', 'class_weights', 'peak_memory_mib', 'wallclock_s',
                  'test_accuracy', 'test_balanced_accuracy', 'test_macro_f1', 'test_ece', 'test_brier', 'test_auc',
                  'test_risk_coverage_auc', 'test_risk_at_coverage_80', 'test_risk_at_coverage_90',
                  'test_risk_at_coverage_95', 'test_temperature', 'error']


def normalize_record(record):
    return {k: record.get(k, float('nan')) for k in RECORD_COLUMNS}


def run_id_for(model_name, stage, key):
    return f'{stage}_{model_name.replace("/", "_")}_{key_name(key)}_{SPLIT_MODE}'


def _key_fields(key):
    return {'protocol': PROTOCOL, 'held_out_mag': key[0], 'split_mode': SPLIT_MODE, 'seed': key[1]}


def _finish(run_dir, record, test_logits, test_labels, test_idx, temperature, full_logits, full_idx, extra=None):
    """Write the per-run outputs. full_* are the logits over EVERY test image, used by the pipeline; their
    subtype labels and magnifications are stored too, so the analysis notebook needs no manifest."""
    _, test_probs = compute_all_metrics(test_logits, test_labels, temperature)
    full_idx = np.asarray(full_idx, dtype=int)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=test_labels, probs=test_probs,
                        manifest_index=test_idx, full_test_logits=full_logits, full_test_index=full_idx,
                        full_test_subtype_label=MANIFEST.subtype_label.to_numpy()[full_idx],
                        full_test_magnification=MANIFEST.magnification.to_numpy()[full_idx],
                        temperature=np.array([temperature]))
    if extra is not None:
        pd.DataFrame(extra).to_csv(run_dir / 'epochs.csv', index=False)
    Path(run_dir, 'record.json').write_text(json.dumps(record, indent=2, default=str))


def train_stage_model(model_name, stage, key):
    seed_everything(key[1])
    rows = stage_rows(key, stage)
    k = len(STAGES[stage]['classes'])
    run_id = run_id_for(model_name, stage, key)
    run_dir = Path(OUTPUT, 'runs', run_id); run_dir.mkdir(parents=True, exist_ok=True)
    weights = class_weights(rows['train'].y, k) if CLASS_WEIGHTING == 'inverse_frequency' else np.ones(k)
    loss_fn = nn.CrossEntropyLoss(weight=torch.as_tensor(weights, dtype=torch.float32, device=DEVICE))
    model = build_from_scratch(model_name, k).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    train_loader, val_loader = _loader(rows['train'], True), _loader(rows['val'], False)
    best, best_ckpt, log = -1.0, run_dir / 'best.pt', []
    _reset_peak(); start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train(); running = 0.0
        for x, y, _ in train_loader:
            optim.zero_grad(set_to_none=True)
            loss = loss_fn(model(x.to(DEVICE, non_blocking=True)), y.to(DEVICE, non_blocking=True))
            loss.backward(); optim.step()
            running += float(loss.detach()) * len(y)
        vl, vy, _ = logits_labels_index(model, val_loader)
        vm, _ = compute_all_metrics(vl, vy)
        log.append({'epoch': epoch, 'train_loss': running / len(train_loader.dataset),
                    **{f'val_{a}': b for a, b in vm.items()}})
        score = vm['macro_f1'] if len(vy) else float(epoch)          # no validation images: keep the last epoch
        if score > best:
            best = score; torch.save(model.state_dict(), best_ckpt)
    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    vl, vy, _ = logits_labels_index(model, val_loader)
    temperature = fit_temperature(vl, vy) if (TEMPERATURE_SCALE and len(vy)) else 1.0
    tl, ty, ti = logits_labels_index(model, _loader(rows['test'], False))
    fl, _, fi = logits_labels_index(model, _loader(full_test_rows(key), False))
    tm, _ = compute_all_metrics(tl, ty, temperature)
    record = normalize_record({'run_id': run_id, 'branch': 'subtype_trained', 'model': model_name, 'stage': stage,
                               **_key_fields(key), 'num_classes': k,
                               'n_train': len(rows['train']), 'n_val': len(rows['val']), 'n_test': len(rows['test']),
                               'class_weights': np.round(weights, 4).tolist(), 'peak_memory_mib': _peak_mib(),
                               'wallclock_s': time.perf_counter() - start, **{f'test_{a}': b for a, b in tm.items()}})
    _finish(run_dir, record, tl, ty, ti, temperature, fl, fi, extra=log)
    del model; torch.cuda.empty_cache()
    return record


def train_stage_probe(stage, key):
    seed_everything(key[1])
    rows = stage_rows(key, stage)
    k = len(STAGES[stage]['classes'])
    run_id = run_id_for(FM_NAME, stage, key)
    run_dir = Path(OUTPUT, 'runs', run_id); run_dir.mkdir(parents=True, exist_ok=True)
    X = {s: torch.as_tensor(FM_FEATS[rows[s].name_idx.to_numpy()]) for s in rows}
    Y = {s: torch.as_tensor(rows[s].y.to_numpy(), dtype=torch.long) for s in rows}
    weights = class_weights(rows['train'].y, k) if CLASS_WEIGHTING == 'inverse_frequency' else np.ones(k)
    loss_fn = nn.CrossEntropyLoss(weight=torch.as_tensor(weights, dtype=torch.float32, device=DEVICE))
    probe = LinearProbe(FM_DIM, k).to(DEVICE)
    optim = torch.optim.AdamW(probe.parameters(), lr=PROBE_LR, weight_decay=WEIGHT_DECAY)
    best, best_state = -1.0, None
    _reset_peak(); start = time.perf_counter()
    for epoch in range(PROBE_EPOCHS):
        probe.train()
        perm = torch.randperm(len(X['train']))
        for i in range(0, len(perm), PROBE_BATCH):
            b = perm[i:i + PROBE_BATCH]
            optim.zero_grad(set_to_none=True)
            loss_fn(probe(X['train'][b].to(DEVICE)), Y['train'][b].to(DEVICE)).backward(); optim.step()
        probe.eval()
        with torch.inference_mode():
            vm, _ = compute_all_metrics(probe(X['val'].to(DEVICE)).cpu().numpy(), Y['val'].numpy())
        score = vm['macro_f1'] if len(Y['val']) else float(epoch)     # no validation images: keep the last epoch
        if score > best:
            best = score; best_state = {a: b.detach().clone() for a, b in probe.state_dict().items()}
    probe.load_state_dict(best_state); probe.eval()
    full = full_test_rows(key)
    with torch.inference_mode():
        vl = probe(X['val'].to(DEVICE)).cpu().numpy()
        tl = probe(X['test'].to(DEVICE)).cpu().numpy()
        fl = probe(torch.as_tensor(FM_FEATS[full.name_idx.to_numpy()]).to(DEVICE)).cpu().numpy()
    temperature = fit_temperature(vl, Y['val'].numpy()) if (TEMPERATURE_SCALE and len(Y['val'])) else 1.0
    tm, _ = compute_all_metrics(tl, Y['test'].numpy(), temperature)
    torch.save({'state_dict': {a: b.cpu() for a, b in best_state.items()}, 'feature_dim': int(FM_DIM),
                'num_classes': k, 'fm_name': FM_NAME, 'temperature': float(temperature)}, run_dir / 'probe.pt')
    record = normalize_record({'run_id': run_id, 'branch': 'subtype_fm_probe', 'model': FM_NAME, 'stage': stage,
                               **_key_fields(key), 'num_classes': k,
                               'n_train': len(rows['train']), 'n_val': len(rows['val']), 'n_test': len(rows['test']),
                               'class_weights': np.round(weights, 4).tolist(), 'peak_memory_mib': _peak_mib(),
                               'wallclock_s': time.perf_counter() - start, **{f'test_{a}': b for a, b in tm.items()}})
    _finish(run_dir, record, tl, Y['test'].numpy(), rows['test'].name_idx.to_numpy(), temperature,
            fl, full.name_idx.to_numpy())
    return record


print('Training primitives ready; stages:', {s: STAGES[s]['classes'] for s in STAGE_ORDER})

In [ ]:
# ---------- Run every stage model: resumable, incremental, time-budgeted ----------
# Order: for each seed and encoder the three stages run back to back, so an interrupted commit still leaves
# complete pipelines. A run starts only if it should finish before MAX_RUNTIME_HOURS (Kaggle stops a commit at
# 12 h); what does not fit is left for the next commit, which restores the finished runs (resume cell).
raw_runs_csv = Path(OUTPUT, 'raw_runs.csv')


def _done(run_id):
    return Path(OUTPUT, 'runs', run_id, 'record.json').exists()


def _record(run_id):
    return json.loads(Path(OUTPUT, 'runs', run_id, 'record.json').read_text())


def _append(record):
    row = pd.DataFrame([normalize_record(dict(record))], columns=RECORD_COLUMNS)
    row.to_csv(raw_runs_csv, mode='a', header=not raw_runs_csv.exists(), index=False)


planned = []
for key in SPLIT_KEYS:                                   # (held-out magnification or None, seed)
    for model_name in FROM_SCRATCH_MODELS:
        for stage in STAGE_ORDER:
            planned.append(('subtype_trained', model_name, stage, key, run_id_for(model_name, stage, key)))
    if INCLUDE_FM:
        for stage in STAGE_ORDER:
            planned.append(('subtype_fm_probe', FM_NAME, stage, key, run_id_for(FM_NAME, stage, key)))
done = [p for p in planned if _done(p[4])]
todo = [p for p in planned if not _done(p[4])]
print(f'planned {len(planned)} stage runs; already finished {len(done)}; to run {len(todo)}')
if raw_runs_csv.exists():
    raw_runs_csv.unlink()
for p in done:
    _append(_record(p[4]))

# Expected hours per run on a T4, scaled from the binary LOMO runs by training-set size (stage 1 uses every
# training image; the benign stage about a third of them, the malignant stage about two thirds). Replaced by
# 1.1 x the longest finished run of the same model and stage once one exists.
_STAGE_SHARE = {'detect': 1.0, 'benign_subtype': 0.33, 'malignant_subtype': 0.7}
# medmamba_no_ssm took 24-27 min per binary run against MedMamba's 46-50, hence about 0.55 x MedMamba.
_STAGE1_HOURS = ({'convnext_tiny': 0.8, 'swin_tiny': 0.75, 'medmamba': 1.15, 'medmamba_no_ssm': 0.65, FM_NAME: 0.03}
                 if PROTOCOL == 'lomo' else
                 {'convnext_tiny': 0.9, 'swin_tiny': 0.85, 'medmamba': 1.3, 'medmamba_no_ssm': 0.75, FM_NAME: 0.03})
try:
    import psutil
    KERNEL_START = psutil.Process(os.getpid()).create_time()
except Exception:
    KERNEL_START = time.time()
deadline = KERNEL_START + (MAX_RUNTIME_HOURS - 0.4) * 3600
longest = {}


def _note(record):
    s = (record or {}).get('wallclock_s')
    if isinstance(s, (int, float)) and s == s and s > 0:
        key = (record['model'], record['stage'])
        longest[key] = max(longest.get(key, 0.0), float(s))


def _expected_s(model_name, stage):
    if (model_name, stage) in longest:
        return 1.1 * longest[(model_name, stage)]
    return _STAGE1_HOURS.get(model_name, 1.5) * _STAGE_SHARE[stage] * 3600


for p in done:
    _note(_record(p[4]))
left = []
t_session = time.perf_counter()
for i, (branch, model_name, stage, key, run_id) in enumerate(todo, 1):
    if time.time() + _expected_s(model_name, stage) > deadline:
        left.append(run_id)
        print(f'[{i}/{len(todo)}] not started: {run_id} needs ~{_expected_s(model_name, stage) / 3600:.2f} h, '
              f'{max(deadline - time.time(), 0) / 3600:.2f} h left')
        continue
    print(f'[{i}/{len(todo)}] {(time.perf_counter() - t_session) / 3600:.2f} h  {run_id}')
    try:
        record = (train_stage_model(model_name, stage, key) if branch == 'subtype_trained'
                  else train_stage_probe(stage, key))
        _append(record); _note(record)
        print(f'    macro-F1 {record["test_macro_f1"]:.3f}  ECE {record["test_ece"]:.3f}  '
              f'({record["wallclock_s"] / 60:.1f} min)')
    except Exception as exc:
        _append({'run_id': run_id, 'branch': branch, 'model': model_name, 'stage': stage,
                 **_key_fields(key), 'error': repr(exc)[:400]})
        print('    FAILED:', repr(exc)[:300])

print(f'Session finished in {(time.perf_counter() - t_session) / 3600:.2f} h.')
if left:
    print(f'Time budget reached: {len(left)} run(s) left. Attach this version as an input and run again.')

In [ ]:
# ---------- The pipeline end to end: stage 1 routes every test image to one stage-2 model ----------
# For each encoder and seed with all three stage runs finished, pipeline_scores() below gives:
#   hard routing (the pipeline as designed): stage 1 predicts benign or malignant, and that branch's subtype
#     model gives the final subtype. Accuracy, balanced accuracy, macro-F1, per-class and per-magnification
#     scores and the 8 x 8 confusion matrix use these predictions.
#   confidence = path probability of the predicted subtype, P(branch) x P(subtype | branch): ECE, AURC,
#     E-AURC and risk at 80% coverage, as for the binary runs. The path probabilities of all 8 subtypes sum
#     to 1 and give the Brier score and the one-vs-rest AUC.
# Stage-1 routing errors are counted separately (benign sent to the malignant branch and the reverse), next to
# wrong subtypes within the correct branch; the three add up to 1 - accuracy. The analysis notebook re-scores
# the stored logits with this same function, so its numbers match these.
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ---- pipeline scoring: the same code in subtype_hierarchical.ipynb and in the analysis notebook ----
import warnings as _pl_warnings
from sklearn.metrics import (balanced_accuracy_score as _pl_bacc, f1_score as _pl_f1, roc_auc_score as _pl_auc,
                             precision_recall_fscore_support as _pl_prfs, confusion_matrix as _pl_cm)
_pl_trapz = getattr(np, 'trapezoid', None) or np.trapz


def pl_softmax(logits, temperature=1.0):
    z = np.asarray(logits, dtype=np.float64) / float(temperature)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def pl_ece(conf, correct, n_bins=15):
    """15 equal-width confidence bins, (lo, hi], as in the training notebooks."""
    edges, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def pl_risk_coverage(conf, correct):
    order = np.argsort(-conf, kind='stable')
    err = (~correct[order]).astype(float)
    k = np.arange(1, len(err) + 1)
    coverage, risk = k / len(err), np.cumsum(err) / k
    return coverage, risk, float(_pl_trapz(risk, coverage))


def pipeline_scores(p1, pb, pm, y8):
    """The two-stage pipeline on N test images.
    p1 (N, 2): stage-1 probabilities (benign, malignant); pb, pm (N, 4): benign- and malignant-subtype
    probabilities; y8 (N,): true subtype, 0-3 benign (A, F, PT, TA) and 4-7 malignant (DC, LC, MC, PC).
    Prediction: stage 1 picks the branch and that branch's model picks the subtype (hard routing).
    Confidence: the path probability of the predicted subtype, P(branch) x P(subtype | branch).
    The path probabilities of all 8 subtypes sum to 1 and give the Brier score and one-vs-rest AUC.
    Returns (scores, per-class arrays, 8 x 8 confusion, predictions)."""
    y8 = np.asarray(y8, dtype=int)
    n = len(y8)
    coarse, true_coarse = p1.argmax(axis=1), (y8 >= 4).astype(int)
    pred = np.where(coarse == 0, pb.argmax(axis=1), 4 + pm.argmax(axis=1))
    p8 = np.hstack([p1[:, :1] * pb, p1[:, 1:] * pm])
    conf, correct = p8[np.arange(n), pred], pred == y8
    _, risk, aurc = pl_risk_coverage(conf, correct)
    err = 1.0 - correct.mean()
    routed = coarse == true_coarse
    with _pl_warnings.catch_warnings():
        _pl_warnings.simplefilter('ignore')            # a subtype absent from a small test set
        bacc = _pl_bacc(y8, pred)
        try:
            auc = _pl_auc(y8, p8, multi_class='ovr', average='macro', labels=list(range(8)))
        except ValueError:
            auc = float('nan')
    prec, rec, f1, sup = _pl_prfs(y8, pred, labels=list(range(8)), zero_division=0)
    scores = {
        'n_test': n, 'accuracy': float(correct.mean()), 'balanced_accuracy': float(bacc),
        'macro_f1': float(_pl_f1(y8, pred, average='macro', labels=list(range(8)), zero_division=0)),
        'ece': pl_ece(conf, correct), 'brier': float(((p8 - np.eye(8)[y8]) ** 2).sum(axis=1).mean()),
        'auc_ovr': float(auc), 'aurc': aurc,
        # E-AURC = AURC minus the lowest AURC reachable at this error rate (Geifman et al., ICLR 2019)
        'e_aurc': float(aurc - (err + (1 - err) * np.log(1 - err))) if err < 1 else float('nan'),
        'risk_at_coverage_80': float(risk[int(0.8 * n) - 1]) if n >= 2 else float('nan'),
        'stage1_accuracy': float(routed.mean()),
        # the three error sources add up to 1 - accuracy
        'error_benign_to_malignant': float(((true_coarse == 0) & (coarse == 1)).mean()),
        'error_malignant_to_benign': float(((true_coarse == 1) & (coarse == 0)).mean()),
        'error_wrong_subtype': float((routed & ~correct).mean()),
        'missed_malignant_rate': float(((true_coarse == 1) & (coarse == 0)).sum() / max((true_coarse == 1).sum(), 1)),
        'subtype_accuracy_when_routed_correctly': float(correct[routed].mean()) if routed.any() else float('nan'),
        'accuracy_soft_argmax': float((p8.argmax(axis=1) == y8).mean())}
    per_class = {'precision': prec, 'recall': rec, 'f1': f1, 'support': sup}
    return scores, per_class, _pl_cm(y8, pred, labels=list(range(8))), pred

rcParams.update({'font.family': 'serif', 'font.serif': ['DejaVu Serif', 'Times New Roman'], 'font.size': 10,
                 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.08})
Path(OUTPUT, 'figures').mkdir(exist_ok=True)
ALL_SUBTYPES = BENIGN_SUBTYPES + MALIGNANT_SUBTYPES              # global labels 0..7
_encoders = FROM_SCRATCH_MODELS + ([FM_NAME] if INCLUDE_FM else [])
_DISPLAY = {'convnext_tiny': 'ConvNeXt-T', 'swin_tiny': 'Swin-T', 'medmamba': 'MedMamba',
            'medmamba_no_ssm': 'MedMamba w/o SS2D', FM_NAME: 'Phikon (frozen FM)'}


def _load_stage(model_name, stage, key):
    d = np.load(Path(OUTPUT, 'runs', run_id_for(model_name, stage, key), 'test_outputs.npz'))
    return pl_softmax(d['full_test_logits'], float(d['temperature'][0])), d['full_test_index']


res_rows, class_rows, mag_rows, confusions = [], [], [], {}
for model_name in _encoders:
    for key in SPLIT_KEYS:
        seed, held = key[1], (key[0] if key[0] is not None else 'pooled')
        if not all(_done(run_id_for(model_name, s, key)) for s in STAGE_ORDER):
            continue
        p1, idx = _load_stage(model_name, 'detect', key)
        pb, idx_b = _load_stage(model_name, 'benign_subtype', key)
        pm, idx_m = _load_stage(model_name, 'malignant_subtype', key)
        assert np.array_equal(idx, idx_b) and np.array_equal(idx, idx_m), 'stage outputs are not aligned'
        truth = MANIFEST.loc[idx]
        y8, mag = truth.subtype_label.to_numpy(), truth.magnification.to_numpy()
        scores, per_class, cm, pred_hard = pipeline_scores(p1, pb, pm, y8)

        row = {'model': _DISPLAY.get(model_name, model_name), 'protocol': PROTOCOL, 'held_out_mag': held,
               'seed': seed, 'split_mode': SPLIT_MODE, **scores}
        res_rows.append(row)
        for c, sub in enumerate(ALL_SUBTYPES):
            class_rows.append({'model': row['model'], 'held_out_mag': held, 'seed': seed, 'subtype': sub,
                               'name': SUBTYPE_NAMES[sub], **{k: v[c] for k, v in per_class.items()}})
        for mg in sorted(set(mag)):
            k = mag == mg
            mag_rows.append({'model': row['model'], 'held_out_mag': held, 'seed': seed, 'magnification': int(mg),
                             'n': int(k.sum()),
                             'accuracy': accuracy_score(y8[k], pred_hard[k]),
                             'macro_f1': f1_score(y8[k], pred_hard[k], average='macro', labels=list(range(8)),
                                                  zero_division=0)})
        confusions.setdefault(row['model'], np.zeros((8, 8), dtype=int))
        confusions[row['model']] += cm

if not res_rows:
    print('No encoder has all three stages finished yet; the end-to-end evaluation runs once they have.')
    hierarchical = None
else:
    hierarchical = pd.DataFrame(res_rows)
    hierarchical.to_csv(Path(OUTPUT, 'hierarchical_results.csv'), index=False)
    pd.DataFrame(class_rows).to_csv(Path(OUTPUT, 'hierarchical_per_class.csv'), index=False)
    pd.DataFrame(mag_rows).to_csv(Path(OUTPUT, 'hierarchical_per_magnification.csv'), index=False)
    np.savez_compressed(Path(OUTPUT, 'hierarchical_confusion.npz'), **{k.replace(' ', '_'): v for k, v in confusions.items()})

    keys = ['accuracy', 'balanced_accuracy', 'macro_f1', 'stage1_accuracy', 'error_benign_to_malignant',
            'error_malignant_to_benign', 'error_wrong_subtype', 'ece', 'brier', 'aurc', 'risk_at_coverage_80']
    summ = hierarchical.groupby(['model', 'held_out_mag'])[keys].agg(['mean', 'std'])
    print(f'End-to-end pipeline, 8 subtypes ({PROTOCOL}, {SPLIT_MODE}-level split; mean and SD over seeds):')
    print(summ.round(3).to_string())
    per_class = pd.DataFrame(class_rows).pivot_table(index='subtype', columns='model', values='f1', aggfunc='mean')
    print('\nPer-subtype F1 (mean over seeds):')
    print(per_class.reindex(ALL_SUBTYPES).round(3).to_string())
    per_mag = pd.DataFrame(mag_rows).pivot_table(index='magnification', columns='model', values='macro_f1', aggfunc='mean')
    print('\nMacro-F1 per magnification (mean over seeds):')
    print(per_mag.round(3).to_string())

    # ---- 8 x 8 confusion matrices, summed over seeds, row-normalised ----
    names = list(confusions)
    fig, axes = plt.subplots(1, len(names), figsize=(5.6 * len(names), 5.4), squeeze=False)
    for ax, name in zip(axes[0], names):
        cm = confusions[name]
        cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
        for r in range(8):
            for c in range(8):
                ax.text(c, r, f'{cmn[r, c]:.2f}', ha='center', va='center', fontsize=7,
                        color='white' if cmn[r, c] > 0.6 else 'black')
        ax.axhline(3.5, color='#B00020', linewidth=1.2); ax.axvline(3.5, color='#B00020', linewidth=1.2)
        ax.set_xticks(range(8)); ax.set_xticklabels(ALL_SUBTYPES); ax.set_yticks(range(8)); ax.set_yticklabels(ALL_SUBTYPES)
        ax.set_xlabel('Predicted subtype'); ax.set_ylabel('True subtype')
        ax.set_title(f'{name}\n(summed over {len(SPLIT_KEYS)} run(s); n = {cm.sum()})', fontsize=11, fontweight='bold')
    _where = (f"held-out {', '.join(f'{m}x' for m in HELD_OUT_MAGS)}" if PROTOCOL == 'lomo' else 'pooled magnifications')
    fig.suptitle(f'Two-stage pipeline, end to end ({_where}, {SPLIT_MODE}-level split). Red lines separate benign '
                 f'(A, F, PT, TA) from malignant (DC, LC, MC, PC); off-block cells are stage-1 routing errors.',
                 fontsize=11.5, fontweight='bold', y=1.03)
    fig.tight_layout()
    fig.savefig(Path(OUTPUT, 'figures', 'hierarchical_confusion.png'), dpi=300)
    fig.savefig(Path(OUTPUT, 'figures', 'hierarchical_confusion.pdf'))
    plt.show()

# ---- every stage on its own (from raw_runs.csv) ----
if raw_runs_csv.exists():
    rr = pd.read_csv(raw_runs_csv)
    rr = rr[rr.error.isna()] if 'error' in rr.columns else rr
    if len(rr):
        rr['model'] = rr.model.map(lambda m: _DISPLAY.get(m, m))
        stage_summary = rr.groupby(['stage', 'model'])[['test_accuracy', 'test_macro_f1', 'test_ece',
                                                        'test_risk_at_coverage_80']].agg(['mean', 'std'])
        stage_summary.to_csv(Path(OUTPUT, 'stage_summary.csv'))
        print('\nEach stage on its own test images (mean and SD over seeds):')
        print(stage_summary.reindex(STAGE_ORDER, level=0).round(3).to_string())
print(f'\nOutputs are in {OUTPUT}; download them from the Output panel.')